# Model Development Journey: Systematic Bitcoin Accumulation

This notebook documents the iterative quantitative research process used to develop our final Bitcoin accumulation strategy. 

Our goal is to systematically outperform a naive Dollar Cost Averaging (DCA) baseline by dynamically adjusting our daily purchase weights based on on-chain valuation (MVRV), momentum, and market regime (200-day moving average).

We will walk through each major iteration of the model, explaining the mathematical decisions, the bugs we encountered, and the breakthroughs that led to our final >60% score.

In [ ]:
import sys
from pathlib import Path

# Add the project root to the Python path so we can import the template module
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from template.prelude_template import load_data, compute_cycle_spd
from template.model_development_template import _clean_array, allocate_sequential_stable
from template.backtest_template import (
    create_performance_comparison_chart,
    create_excess_percentile_distribution,
    create_win_loss_comparison,
    create_cumulative_performance
)

import example_1.model_development_example_1 as ex1

# Load core data
print("Loading Bitcoin data...")
df_btc = load_data()
print(f"Data loaded: {df_btc.index.min().date()} to {df_btc.index.max().date()}")

## 0. The Benchmark (Example 1)

Before developing our own model, we must establish a benchmark. The provided `Example 1` model uses MVRV, the 200-day MA, and Polymarket sentiment. We will run it here to establish the score and mean excess Sats-Per-Dollar (SPD) that we need to beat.

In [ ]:
print("Precomputing features for Example 1...")
features_ex1 = ex1.precompute_features(df_btc)

def wrapper_ex1(df_window):
    start_date = df_window.index.min()
    end_date = df_window.index.max()
    return ex1.compute_window_weights(
        features_df=features_ex1,
        start_date=start_date,
        end_date=end_date,
        current_date=end_date
    )

print("Running backtest for Example 1 Benchmark...")
df_spd_ex1 = compute_cycle_spd(
    dataframe=df_btc,
    strategy_function=wrapper_ex1,
    features_df=features_ex1,
    start_date="2018-01-01",
    end_date="2025-12-31"
)

wins_ex1 = (df_spd_ex1["dynamic_sats_per_dollar"] > df_spd_ex1["uniform_sats_per_dollar"]).sum()
win_rate_ex1 = wins_ex1 / len(df_spd_ex1)
excess_ex1 = (df_spd_ex1["dynamic_sats_per_dollar"] - df_spd_ex1["uniform_sats_per_dollar"]) / df_spd_ex1["uniform_sats_per_dollar"]
score_ex1 = (win_rate_ex1 * 0.5) + (max(0, excess_ex1.median()) * 0.5)

print(f"--- Results: Example 1 Benchmark ---")
print(f"Win Rate: {win_rate_ex1:.2%}")
print(f"Score: {score_ex1:.2%}")
print(f"Mean Excess SPD: {excess_ex1.mean():.2%}")
print(f"Median Excess SPD: {excess_ex1.median():.2%}\n")

In [ ]:
# Precompute features once to speed up iterative testing for our models
def precompute_all_features(df):
    price = df["PriceUSD_coinmetrics"].loc["2010-07-18":].copy()
    
    # MVRV Z-Score
    mvrv = df["CapMVRVCur"].loc[price.index]
    mvrv_mean = mvrv.rolling(365, min_periods=180).mean()
    mvrv_std = mvrv.rolling(365, min_periods=180).std()
    mvrv_zscore = ((mvrv - mvrv_mean) / mvrv_std).fillna(0).clip(-4, 4)
    
    # Momentum
    roi_30d = price.pct_change(30).fillna(0).clip(-1, 1)
    roi_1yr = price.pct_change(365).fillna(0).clip(-2, 5)
    
    # Volatility
    daily_ret = price.pct_change().fillna(0)
    vol_30d = daily_ret.rolling(30, min_periods=15).std().fillna(0)
    volatility_pct = vol_30d.rolling(365, min_periods=180).apply(
        lambda x: (x.iloc[-1] > x[:-1]).sum() / max(len(x) - 1, 1) if len(x) > 1 else 0.5, raw=False
    ).fillna(0.5)
    
    # Regime & Trajectory
    ma_200 = price.rolling(200, min_periods=100).mean()
    price_vs_ma = (price / ma_200) - 1.0
    price_vs_ma = price_vs_ma.fillna(0).clip(-0.8, 2.0)
    mvrv_gradient = mvrv_zscore.diff(30).fillna(0).clip(-3, 3)
    
    features = pd.DataFrame({
        "PriceUSD_coinmetrics": price,
        "mvrv_zscore": mvrv_zscore,
        "roi_30d": roi_30d,
        "roi_1yr": roi_1yr,
        "volatility_pct": volatility_pct,
        "price_vs_ma": price_vs_ma,
        "mvrv_gradient": mvrv_gradient,
        "fed_uncertainty": 0.5 # Placeholder for Polymarket
    }, index=price.index)
    
    signal_cols = ["mvrv_zscore", "roi_30d", "roi_1yr", "volatility_pct", "price_vs_ma", "mvrv_gradient", "fed_uncertainty"]
    features[signal_cols] = features[signal_cols].shift(1).fillna(0)
    return features

print("Precomputing features for our models...")
features_df = precompute_all_features(df_btc)

# Evaluation Helper
def evaluate_model(multiplier_func, name="Model"):
    def compute_weights(df):
        if df.empty: return pd.Series(dtype=float)
        
        base = np.ones(len(df)) / len(df)
        z = _clean_array(df["mvrv_zscore"].values)
        r30 = _clean_array(df["roi_30d"].values)
        r1y = _clean_array(df["roi_1yr"].values)
        v = _clean_array(df["volatility_pct"].values)
        pma = _clean_array(df["price_vs_ma"].values)
        g = _clean_array(df["mvrv_gradient"].values)
        f = _clean_array(df["fed_uncertainty"].values)
        
        dyn = multiplier_func(z, r30, r1y, v, pma, g, f)
        raw = base * dyn
        weights = allocate_sequential_stable(raw, len(df), None)
        return pd.Series(weights, index=df.index)

    print(f"Running backtest for {name}...")
    df_spd = compute_cycle_spd(
        dataframe=df_btc,
        strategy_function=compute_weights,
        features_df=features_df,
        start_date="2018-01-01",
        end_date="2025-12-31"
    )
    
    wins = (df_spd["dynamic_sats_per_dollar"] > df_spd["uniform_sats_per_dollar"]).sum()
    win_rate = wins / len(df_spd)
    excess = (df_spd["dynamic_sats_per_dollar"] - df_spd["uniform_sats_per_dollar"]) / df_spd["uniform_sats_per_dollar"]
    score = (win_rate * 0.5) + (max(0, excess.median()) * 0.5)
    
    print(f"--- Results: {name} ---")
    print(f"Win Rate: {win_rate:.2%}")
    print(f"Score: {score:.2%}")
    print(f"Mean Excess SPD: {excess.mean():.2%}")
    print(f"Median Excess SPD: {excess.median():.2%}\n")
    return df_spd

## 1. The Starting Point (Baseline)

Our initial model relied primarily on the MVRV Z-score and basic return momentum. While it achieved a win rate of ~60%, its mean excess Sats-Per-Dollar (SPD) was relatively low. It was buying more during cheap periods, but it wasn't aggressive enough during structural market bottoms.

In [ ]:
def multiplier_v1(z, r30, r1y, v, pma, g, f):
    # Simple MVRV + Momentum
    mvrv_signal = -z
    mvrv_boost = np.where(z < -1.5, (z + 1.5)**2, 0)
    mom_signal = (r30 * 1.5) + (r1y * 0.25)
    
    combined = ((mvrv_signal + mvrv_boost) * 0.75) + (mom_signal * 0.25)
    
    dampener = np.where(v > 0.8, 1.0 - 0.4 * ((v - 0.8) / 0.2), 1.0)
    
    adjustment = np.clip(combined * 3.5, -4, 10)
    multiplier = np.exp(adjustment) * dampener
    return np.where(np.isfinite(multiplier), multiplier, 1.0)

df_spd_v1 = evaluate_model(multiplier_v1, "V1: Baseline")

## 2. Iteration 1: Regime & Trajectory (The "Sniper" Model)

Based on our EDA, we knew the 200-day Moving Average was a powerful regime filter, and the *change* in MVRV (gradient) was crucial for confirming bottoms. We added these features. 

**The Result:** The model became a "sniper". It saved all its capital for the absolute bottom day of a crash (spiking to 95x normal allocation), but starved the rest of the bear market. The score dropped slightly because it was too concentrated.

In [ ]:
def multiplier_v2(z, r30, r1y, v, pma, g, f):
    mvrv_signal = -z
    deep_value = z < -1.5
    improving = g > 0
    
    mvrv_boost = np.where(deep_value, (z + 1.5)**2, 0)
    bottom_confirmation = np.where(deep_value & improving, mvrv_boost * 0.75, 0)
    
    # Multiplicative regime modifier
    regime_multiplier = np.where(pma < 0, 1.0 + np.abs(pma), np.maximum(0.05, 1.0 - pma))
    
    combined = ((mvrv_signal + mvrv_boost + bottom_confirmation) * 0.75) + ((r30 * 1.5 + r1y * 0.25) * 0.25)
    combined = combined * regime_multiplier
    
    adjustment = np.clip(combined * 3.5, -4, 10)
    multiplier = np.exp(adjustment)
    return np.where(np.isfinite(multiplier), multiplier, 1.0)

df_spd_v2 = evaluate_model(multiplier_v2, "V2: The Sniper")

## 3. Iteration 2: The Multiplicative Bug

To fix the "sniper" issue, we attempted to implement *Regime-Conditional Blending*—weighting value heavily in bear markets and momentum heavily in bull markets. 

**The Bug:** We kept the regime modifier as a multiplier (`combined * regime_multiplier`). At the peak of a bull market, the base signal is deeply negative (e.g., `-3.0`, meaning "do not buy"). Multiplying this by a small regime fraction (e.g., `0.05`) shrank the signal to `-0.15`. When exponentiated, `exp(-0.15)` is close to `1.0x`. We accidentally forced the model to buy at normal DCA rates right at the market top! Performance plummeted.

In [ ]:
def multiplier_v3(z, r30, r1y, v, pma, g, f):
    value_score = -z + np.where((z < -1.0) & (g > 0), np.abs(z)**1.5, 0)
    mom_score = (r30 * 2.0) + (r1y * 0.5)
    
    is_bear = pma < 0
    combined = np.where(is_bear, (value_score * 0.8) + (mom_score * 0.2), (value_score * 0.4) + (mom_score * 0.6))
    
    # The Bug: Multiplicative scaling of a signal that crosses zero
    regime_multiplier = np.where(is_bear, 1.0 + np.abs(pma) * 2.0, np.maximum(0.2, 1.0 - pma * 0.75))
    combined = combined * regime_multiplier
    
    adjustment = np.clip(combined * 2.0, -3, 5)
    multiplier = np.exp(adjustment)
    return np.where(np.isfinite(multiplier), multiplier, 1.0)

df_spd_v3 = evaluate_model(multiplier_v3, "V3: Multiplicative Bug")

## 4. Iteration 3: Log-Space Additivity (The Breakthrough)

To fix the bug, we realised that because we exponentiate the final output, we must apply the regime shift **additively in log-space**. 

Because `exp(A + B) = exp(A) * exp(B)`, adding a penalty in log-space correctly acts as a true multiplier in real-space. This preserves the direction and magnitude of the "do not buy" signal at market tops, while aggressively boosting accumulation at market bottoms.

**The Result:** A massive breakthrough. The score jumped past 60%, with a win rate of 64% and a median relative improvement of >20%.

In [ ]:
def multiplier_v4(z, r30, r1y, v, pma, g, f):
    value_score = -z
    deep_value = z < -1.0
    improving = g > 0
    
    value_boost = np.where(deep_value, np.abs(z) - 1.0, 0)
    confirmation_boost = np.where(deep_value & improving, value_boost * 1.0, 0)
    value_score = value_score + value_boost + confirmation_boost

    mom_score = r30 * 2.0
    base_signal = (value_score * 0.8) + (mom_score * 0.2)

    # The Fix: Additive shift in log-space
    regime_shift = np.where(pma < 0, np.abs(pma) * 2.0, -pma * 3.0)
    combined = base_signal + regime_shift

    dampener = np.where(v > 0.85, 1.0 - 0.5 * ((v - 0.85) / 0.15), 1.0)
    
    adjustment = np.clip(combined * 2.5 * dampener, -4.0, 4.5)
    multiplier = np.exp(adjustment)
    return np.where(np.isfinite(multiplier), multiplier, 1.0)

df_spd_v4 = evaluate_model(multiplier_v4, "V4: Log-Space Additivity (Final Model)")

## 5. Iteration 4: The Polymarket Null Result

Finally, we tested integrating Polymarket Fed Uncertainty. Our EDA suggested high macro uncertainty predicts lower forward volatility, which could allow us to safely buy during chaotic periods.

**The Result:** The impact was negligible (score dropped slightly to ~60.06%). Why? The Polymarket odds history only covers 2023-2026, and the volatility dampener is a rare trigger. As our EDA warned, prediction market data proved coincident rather than incrementally predictive over MVRV. 

In the interest of model parsimony and empirical rigour, we discarded this feature and accepted the null result. **V4 remains our final model.**

## 6. Final Visualisations

To conclude, we generate the standard suite of performance charts for our final V4 model, demonstrating its consistency and magnitude of outperformance against uniform DCA.

In [ ]:
import os
from pathlib import Path
from IPython.display import SVG, display

output_dir = Path("output_notebook")
output_dir.mkdir(exist_ok=True)

print("Generating final performance charts...")
create_performance_comparison_chart(df_spd_v4, str(output_dir))
create_excess_percentile_distribution(df_spd_v4, str(output_dir))
create_win_loss_comparison(df_spd_v4, str(output_dir))
create_cumulative_performance(df_spd_v4, str(output_dir))

### Performance Comparison
This chart shows the distribution of Sats-Per-Dollar (SPD) across all rolling windows.

In [ ]:
display(SVG(filename=str(output_dir / "performance_comparison.svg")))

### Excess Percentile Distribution
This highlights the magnitude of outperformance. A right-skewed distribution indicates the strategy consistently acquires more Bitcoin than the baseline.

In [ ]:
display(SVG(filename=str(output_dir / "excess_percentile_distribution.svg")))

### Win/Loss Comparison
A breakdown of how often the strategy beats the uniform DCA baseline.

In [ ]:
display(SVG(filename=str(output_dir / "win_loss_comparison.svg")))

### Cumulative Performance
Tracking the cumulative advantage of the dynamic strategy over time.

In [ ]:
display(SVG(filename=str(output_dir / "cumulative_performance.svg")))